# eBay Laptops & Notebooks - Data Cleansing

The following dataset contains information about laptops and notebooks, by web scraping the popular e-commerce website
**eBay**.

The dataset contains information about the laptop prices (possible variable of interest), brand, ratings, condition, as well as hardware information (processor, screen size, ram, etc).

<div style="text-align: center;">
<img src="../assets/logos/ebay.png" width="250"/>
</div>

In [1]:
# Importing libraries and setting constants
import polars as pl
from polars import LazyFrame
import re

In [2]:
# Loading the dataset
df = pl.read_csv('../data/ebay_laptops_and_notebooks.csv')

# Top 10 rows from the dataframe
df.head(n=10)

Brand,Price,Rating,Ratings Count,Condition,Seller Note,Processor,Screen Size,Manufacturer Color,Color,Ram Size,SSD Capacity,GPU,Processor Speed,Type,Release Year,Maximum Resolution,Model,OS,Features,Hard Drive Capacity,Country Region Of Manufacture,Storage Type
str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""CHUWI""","""$303.68 """,null,null,"""New: A brand-new, unused, unop…",null,"""Quad Core""","""14 in""",null,"""Gray""","""8 GB""","""1 Used, 1 support Max 1TB M.2…","""Intel Iris Plus Graphics 655""","""Max up to 3.80 GHz""","""Notebook/Laptop""","""2021""","""2160 x 1440""","""CoreBook X""","""Windows 11 Home""","""Backlit Keyboard, Built-in Mi…","""512 GB SSD""","""China""","""SSD (Solid State Drive)"""
"""Dell""","""$399.99 to $634.99""",null,null,"""Very Good - RefurbishedThe ite…","""“AAA PCs is a Microsoft Author…","""Intel Core i7 8th Gen.""","""14 in""",null,"""Black""",null,null,"""Intel UHD Graphics 620""","""4.20 GHz (1.90 GHz Base Freque…","""Notebook/Laptop""","""Refurbished in 2023""","""1920 x 1080""","""Dell Latitude 7490""","""Windows 11 Pro""","""Backlit Keyboard, Bluetooth, B…","""2 TB""",null,"""SSD (Solid State Drive)"""
"""Dell""","""$175.00 """,null,null,"""UsedAn item that has been used…","""“Well kept, fully functional, …","""Intel Core i5-6300U""","""14 in""",null,"""Black""","""16 GB""","""500 GB""","""Intel HD Graphics""","""2.40 GHz""","""Notebook/Laptop""","""2019""","""1920 x 1080""","""Dell Latitude E5470""","""Windows 10 Pro""","""10/100 LAN Card, Backlit Keybo…","""500 GB""","""China""","""SSD (Solid State Drive)"""
"""HP""","""$84.99 """,null,null,"""Good - RefurbishedThe item sho…","""“1-Year Allstate warranty. The…","""Intel Celeron N""","""11.6 in""",null,"""Black""","""4 GB""",null,"""Intel HD Graphics 500""","""2.40 GHz""","""Notebook/Laptop""",null,"""1366 x 768""","""HP Chromebook 11 G6""","""Chrome OS""","""Bluetooth, Built-in Microphone…","""16 GB""",null,"""eMMC"""
"""Dell""","""$101.22 """,null,null,"""Good - RefurbishedThe item sho…","""“Laptops is tested & fully wor…","""Intel Core i5 6th Gen.""","""Minimum 12.5""""",null,null,"""8 GB""","""256 GB""","""Integrated""","""Minimum 1.40 GHz""","""Notebook/Laptop""","""2015""","""1366 x 768""","""Various Models""","""Windows 10""","""10/100 LAN Card, Built-in Micr…","""NO HDD""",null,"""SSD (Solid State Drive)"""
"""Acer""","""$49.99 ""","""4.5 out of 5 stars""",193,null,null,"""Intel Celeron""","""11.6 in""","""Black""","""Black""","""4 GB""","""16 GB""",null,"""1.60 GHz""","""Laptop""","""2017""","""1366 x 768""","""Chromebook C738T-C44Z""","""Chrome OS""","""Touchscreen, Bluetooth""",null,null,"""SSD (Solid State Drive)"""
"""HP""","""$39.59 ""","""4.5 out of 5 stars""",23,null,null,"""Intel Celeron""","""11.6 in""",null,"""Black""","""4 GB""","""16GB""",null,"""1.60 GHz""","""Notebook/Laptop""","""2017""",null,"""HP Chromebook 11 G5""","""Chrome OS""","""Bluetooth, Built-in Webcam""","""16 GB""",null,"""SSD (Solid State Drive)"""
"""Acer""","""$34.99 ""","""5 out of 5 stars""",20,null,null,"""Intel Celeron Dual-Core""","""11.6 inin""",null,"""Gray, Granite Gray""","""2GB""",null,null,"""1.00 GHz.4ghz""","""Netbook""",null,null,"""Chromebook C720-2848""","""Chrome OS""",null,"""16GB""",null,null
"""Dell""","""$279.99 """,null,null,"""Excellent - Refurbished: The i…",null,"""i7 7th""","""14 in""",null,"""Black""","""16 GB""","""500 GB""",null,"""2.80 GHz""","""Notebook/Laptop""",null,"""1920 x 1080""","""Dell Latitude 7480""","""win 11 pro""","""Wi-Fi""","""512 GB""",null,"""SSD (Solid State Drive)"""


Observations:

* `Price` variable contains information about the price itself, but also the currency. Also, some prices are in a range format, such as `$399.99 to $634.99` which needs to be handled before converting `Price` into a numerical variable.
* **Several** columns are being treated as `String` (such as `Price`), which might not be adequate.
* **Several** columns have **missing values**, which needs to be treated during further data preprocessing.

In [3]:
# Summary statistics
df.describe()

statistic,Brand,Price,Rating,Ratings Count,Condition,Seller Note,Processor,Screen Size,Manufacturer Color,Color,Ram Size,SSD Capacity,GPU,Processor Speed,Type,Release Year,Maximum Resolution,Model,OS,Features,Hard Drive Capacity,Country Region Of Manufacture,Storage Type
str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""count""","""4024""","""6620""","""286""",286.0,"""6004""","""1442""","""3837""","""3890""","""185""","""2149""","""2112""","""2780""","""3088""","""2669""","""3289""","""787""","""2480""","""3148""","""3040""","""2505""","""2332""","""203""","""2388"""
"""null_count""","""2596""","""0""","""6334""",6334.0,"""616""","""5178""","""2783""","""2730""","""6435""","""4471""","""4508""","""3840""","""3532""","""3951""","""3331""","""5833""","""4140""","""3472""","""3580""","""4115""","""4288""","""6417""","""4232"""
"""mean""",null,null,null,42.479021,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""std""",null,null,null,201.257428,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""","""?""","""$1,002.67 ""","""1 out of 5 stars""",1.0,"""Certified - Refurbished: The i…","""“""Excellent Condition Touch Sc…","""10th Gen Intel Core i5""","""10 in""","""3000""","""Abyss Blue""","""1 GB""",""".""",""".""","""1.00 GHz""","""2 IN 1 CHROMEBOOK""","""0""","""1024 x 480""","""100e Chromebook""","""ANDROID""",""".""",""".""","""Australia""","""."""
"""25%""",null,null,null,2.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,null,null,4.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,null,null,20.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""","""xnote""","""$999.95 ""","""5 out of 5 stars""",1533.0,"""Very Good - RefurbishedThe ite…","""“✅ Refurbished ✅ inkl. 19% MwS…","""unknown""","""none""","""onyx black""","""see pictures""","""up to 32GBGB DDR4""","""up to 1TB SSD""","""‎PowerVR GX6250""","""up to 3.6Ghz""","""Écran numériseur""","""see manufacturer""","""See Title/Description""","""xnote""","""‎Windows 11 Home""","""dvd""","""up to 1 TB""","""Unknown""","""ssd"""


In [4]:
# Data Types
pl.DataFrame(data={
    'column': df.columns,
    'dtype': df.dtypes
})

column,dtype
str,object
"""Brand""",String
"""Price""",String
"""Rating""",String
"""Ratings Count""",Int64
"""Condition""",String
…,…
"""OS""",String
"""Features""",String
"""Hard Drive Capacity""",String


As aforementioned, many variables are currently being treated as `String`. In the case of `Price`, it is a numerical variable which is being incorrectly read as a categorical variable.

## Data Preparation

In [5]:
# Renaming columns: replacing blank spaces for underscores and lowercasing columns
df = df.rename({col: col.lower().replace(' ', '_') for col in df.columns})
print(df.columns)

['brand', 'price', 'rating', 'ratings_count', 'condition', 'seller_note', 'processor', 'screen_size', 'manufacturer_color', 'color', 'ram_size', 'ssd_capacity', 'gpu', 'processor_speed', 'type', 'release_year', 'maximum_resolution', 'model', 'os', 'features', 'hard_drive_capacity', 'country_region_of_manufacture', 'storage_type']


In [6]:
# Nulls per each column in the dataset
df.null_count().unpivot(
    variable_name='variable',
    value_name='null_count',
).sort(
    by='null_count', descending=True,
).with_columns(
    (pl.col('null_count') / df.height).round(2).alias('null_proportion'),
)

variable,null_count,null_proportion
str,u32,f64
"""manufacturer_color""",6435,0.97
"""country_region_of_manufacture""",6417,0.97
"""rating""",6334,0.96
"""ratings_count""",6334,0.96
"""release_year""",5833,0.88
…,…,…
"""processor""",2783,0.42
"""screen_size""",2730,0.41
"""brand""",2596,0.39


Observations:

- Extremely high percentage (85%+) of nulls for columns such as `country_region_of_manufacturer`, `ratings_count` or `release_year`.
- `Price` column has no null values.

## Transforming the `brand` variable
- **Top categories**: The top three brands—Dell (19%), Lenovo (11%), and HP (7%)—represent a combined 37% of the total brands in the dataset. The null values account for 39% of the data, with 2596 entries marked as null. This is quite a large portion of the dataset. The top 4 categories (`null`, `Dell`, `Lenovo` & `HP`) account for 76% of the dataset.
- **Unknown values**: There's an abundance of brands with very low frequencies, or unknown values (like `?` or `Does not apply`).
- **Inconsistent formatting**: Some entries like `Dell Inc` are not captured under the main brand `Dell` due to inconsistent formatting. Similar things occur to other top brands.
- **Multibrand categories**: Some entries represent several brands, like `Dell / HP / Lenovo` or `Apple / LG`

In [7]:
# Taking a look at the frequency of each brand: total count and proportion
df['brand'].value_counts(
    sort=True,
    parallel=True,
).with_columns(
    (pl.col('count') / df.height).round(2).alias('proportion')
)

brand,count,proportion
str,u32,f64
null,2596,0.39
"""Dell""",1273,0.19
"""Lenovo""",715,0.11
"""HP""",494,0.07
"""Acer""",147,0.02
…,…,…
"""Frontier""",1,0.0
"""Dell/HP/Lenovo/Stone""",1,0.0
"""SCSI""",1,0.0


### **Reformatting Brands:**

In this step of our analysis, we're focusing on cleaning up and standardizing the `brand` data to ensure consistency and accuracy. Brands can be listed in many different ways, which can lead to confusion or misinterpretation during analysis. So, we're transforming the brand names into a uniform format that makes it easier to work with.

- **Consistent Formatting:** We ensure all brand names are in lowercase, and we remove any unnecessary spaces or punctuation that could create inconsistencies. (transforming `Dell Inc` into `dell_inc`).
- **Replacing Uncertain Values:** We also handle special cases where the data might have uncertain or irrelevant values (e.g., a "?" for unknown brands), replacing them with more meaningful terms like "unbranded" or "unknown."


In [8]:
# Define a mapping to replace certain placeholder values with more meaningful labels.
brand_replace_map = {
    '?': 'unbranded',  # Replace '?' with 'unbranded' for unknown brands.
    'does_not_apply': 'not_applicable',  # Replace 'does_not_apply' with 'not_applicable' to handle irrelevant data.
}

# Clean and normalize the brand names.
df = df.with_columns(
    pl.col('brand')
    .str.to_lowercase()  # Convert brand names to lowercase to ensure uniformity.
    .str.strip_chars()  # Remove any leading whitespace or special characters.
    .str.strip_chars_end('.')  # Specifically remove any trailing dots at the end of brand names.
    .str.replace_all(' / |\\/', ' or ')  # Replace slashes ("/" or " / ") with ' or ' to standardize multi-brand names.
    .str.replace_all(' ', '_')  # Replace spaces with underscores to ensure no spaces in brand names.
    .replace(brand_replace_map)  # Replace values like '?' or 'does_not_apply' based on the predefined map.
    .fill_null('unknown')  # Fill any missing brand values with 'unknown' to handle missing data.
    .alias('brand_clean')
)

### **Identifying and Mapping Top Brands:**

In this step, we're focusing on identifying the most popular brands in the dataset and ensuring that any variations in the brand names are mapped to the correct top brand.

- **Identifying Top Brands:** We start by filtering out any unknown brand names and then count the occurrences of each brand. From this, we identify the top 5 brands with the highest frequency in the dataset.
- **Pattern Matching:** We create a pattern to search for these top brands in the brand names. If the brand name matches one of the top brands (e.g., includes "dell", "hp", etc.), we recognize it as a valid entry for that brand.
- **Mapping Variations to Top Brands:** We then apply a function to map variations of the brand names (e.g., "dell_epsilon" or "hp_dell") to the main brand names (e.g., "dell" or "hp"). This ensures that similar or ambiguous entries are standardized to the correct top brand.

By applying these steps, we ensure that any variations or inconsistencies in the brand names are correctly mapped to the top brands, improving the consistency and accuracy of our analysis.


In [9]:
# Identify the top 5 most frequent brands in the dataset.
top_brands = df.filter(pl.col('brand_clean').ne('unknown')) \
    .group_by('brand_clean') \
    .agg(pl.len().alias('count')) \
    .sort('count', descending=True) \
    .head(5) \
    .select('brand_clean') \
    .to_series()

# Create a regex pattern to match brand names followed by an underscore (_).
is_a_top_brand_string_pattern = "|".join(re.escape(color) for color in top_brands + '_')


# Function to extract the actual brand from a string.
def return_actual_brand_from_string(string: str, brand_list: list[str] = top_brands) -> str:
    """
    Extracts the first matching brand from a given string.

    Args:
        string (str): The input string containing the brand.
        brand_list (list): List of top brands to check against.

    Returns:
        str: The first matched brand if found; otherwise, the original string.
    """
    for brand in brand_list:
        if brand + '_' in string:
            return brand
    return string


# Apply the brand extraction function to standardize brand names.
df = df.with_columns(
    pl
    # Check if the brand contains any of the top brands and isn't a mixed brand like "dell_or_hp".
    .when(
        pl.col('brand_clean').str.contains(is_a_top_brand_string_pattern) &
        ~pl.col('brand_clean').str.contains('_or_')
    )
    # If the condition is met, map the brand name to its standardized version.
    .then(pl.col('brand_clean').map_elements(return_actual_brand_from_string, return_dtype=pl.Utf8))
    # Otherwise, keep the original brand name.
    .otherwise(pl.col('brand_clean'))
    # Rename the transformed column as 'brand_clean'.
    .alias('brand_clean')
)


In [10]:
# Taking another look at the frequency of each brand: total count and proportion
df['brand_clean'].value_counts().with_columns(
    (pl.col('count') / df.height).round(2).alias('proportion')
).sort(by='count', descending=True)

brand_clean,count,proportion
str,u32,f64
"""unknown""",2596,0.39
"""dell""",1300,0.2
"""lenovo""",720,0.11
"""hp""",494,0.07
"""acer""",150,0.02
…,…,…
"""orbitkey""",1,0.0
"""unbranded_or_generic""",1,0.0
"""insignia""",1,0.0


Observations:
- **More records assigned to `dell`**: `dell` has more than 20 new records. (from `1273` to `1300`). Likely due to previously fragmented brand variations (e.g., `dell_inc`, `dell_xps`) now being mapped correctly to `dell`.
- **Consistent top brands**: The overall ranking of top brands did not change. The counts for `acer` (150), `lg` (125), `fujitsu_siemens` (123), `samsung` (122), `microsoft` (119), and AUO (114) remained stable, indicating that these brands were either already well-represented or had fewer naming inconsistencies.

## Transforming the `price` variable

- **Multiple info**: By analyzing the `price` variable, we realize it also has information about the currency (via symbols like `$`).
- **Inconsistent format**: The cost of a laptop might be fixed (e.g: `880`) or a range (e.g: `from 500 to 999`)
- **Bad typing**: `price` is currently a categorical (`String`) variable, due to commas, whitespaces and symbols. To use this data effectively for statistical models, we **MUST CONVERT** it into **NUMERICAL**, removing currency symbols and handling text-based variations like ranges.

In [11]:
# Taking a look at the first 5 records from the `price` column, as is
df['price'].head(n=10)

price
str
"""$303.68 """
"""$399.99 to $634.99"""
"""$175.00 """
"""$84.99 """
"""$101.22 """
"""$49.99 """
"""$39.59 """
"""$34.99 """
"""$279.99 """


Observations:
- **Variable format**: Cost of a laptop might be either a range (`$399.99 to $634.99`) or a fixed price (`$399.99 to $634.99`).
- **Wrong typing**: Data is currently of `String` data type, instead of a numerical type, due to the presence of currency symbols and words ('to').

### **Reformatting `price`:**

In this step, we focus on cleaning and standardizing the `price` variable to make it usable for further analysis. Currently, the `price` column contains multiple inconsistencies that need to be addressed.

- **Extracting Currency Symbols:** The dataset contains various currency symbols (such as `$`), which are separated from the numerical values, and assigned to a new variable `currency_clean`.
- **Handling Inconsistent Formats:** Prices appear in different formats, sometimes as a single fixed value (`880`) and other times as a range (`from 500 to 999`). We need to extract the lowest value from these ranges (`min_price_clean`) to ensure uniformity.
- **Ensuring Proper Data Types:** After cleaning, we cast the extracted price values to `Float64`, making them compatible with statistical models and numerical computations.


In [12]:
df = df.with_columns(
    # Extract the currency symbol from the price column
    pl.col('price').str.slice(0, 1).alias('currency_clean'),
    # Removing the currency symbol, commas from the price and trimming whitespace
    pl.col('price')
    .str.slice(1)  # Removing the currency symbol
    .str.replace(',', '')  # Removing commas
    .str.split("to")  # Separating the price into min_price and max_price range
    .list.first()  # Obtaining the min_price, the first element of the list
    .str.strip_chars()  # Remove leading and trailing characters
    .cast(pl.Float64)  # Casting the min_price to float
    .alias('min_price_clean')  # Renaming to reflect that it's the min price in case of ranges
)

# Preview the transformed data by displaying the top 5 rows
df.select([
    'price', 'currency_clean', 'min_price_clean'
]).head(n=10)

price,currency_clean,min_price_clean
str,str,f64
"""$303.68 ""","""$""",303.68
"""$399.99 to $634.99""","""$""",399.99
"""$175.00 ""","""$""",175.0
"""$84.99 ""","""$""",84.99
"""$101.22 ""","""$""",101.22
"""$49.99 ""","""$""",49.99
"""$39.59 ""","""$""",39.59
"""$34.99 ""","""$""",34.99
"""$279.99 ""","""$""",279.99


## Transforming the `condition` variable

By analyzing the `condition` variable, we realize it is composed of two separate items, label and description. It is important to create such columns within the dataset:
- **`condition_label`**: A categorical label which represents the condition of a laptop (e.g: `new`, `used`, `certified_refurbished`)
- **`condition_description`**: A description for the `condition_label` (e.g: `A brand-new, unused, unopened laptop...`)

In [13]:
# Taking a look at the values counts for the `condition` column
df['condition'].value_counts(
    sort=True,
    parallel=True,
).with_columns(
    (pl.col('count') / df.height).round(2).alias('proportion')
)

condition,count,proportion
str,u32,f64
"""New: A brand-new, unused, unop…",1931,0.29
"""Used: An item that has been us…",1785,0.27
"""UsedAn item that has been used…",1009,0.15
null,616,0.09
"""Seller refurbished: The item h…",576,0.09
…,…,…
"""Very Good - Refurbished: The i…",35,0.01
"""Certified - RefurbishedThe ite…",35,0.01
"""Open box: An item in excellent…",26,0.0


Observations:
- **Similar categories**: Labels `UsedAn item that has been used previously` and `Used: An item that has been used previously` seem to contain duplicate or nearly identical information but are represented differently. Similar things occur for other labels, such as `For parts or not working`. These discrepancies may need to be cleaned and consolidated into one label.
- **Bad formatting**: A considerable amount of labels show formatting issues (e.g., "UsedAn item...") or seem to be concatenated with additional descriptions. This would require text processing to standardize the condition labels and improve consistency.

In [14]:
# Creating a dictionary which represents the condition labels and descriptions

condition_replace_map = {
    'New': """A brand-new, unused, unopened, undamaged item in its original packaging.
    Packaging should be the same as what is found in a retail store, unless the item is handmade or was packaged by the manufacturer in non-retail packaging, such as an unprinted box or plastic bag.""",

    'Open box': """An item in excellent, new condition with no wear.
    The item may be missing the original packaging or protective wrapping, or may be in the original packaging but not sealed.
    The item includes original accessories and may be a factory second.""",

    'Certified - Refurbished': """The item is in pristine, like-new condition.
    It has been professionally inspected, cleaned, and refurbished by the manufacturer or a manufacturer-approved vendor to meet manufacturer specifications.
    The item will be in new packaging with original or new accessories.""",

    'Excellent - Refurbished': """The item is in like-new condition, backed by a one-year warranty.
    It has been professionally refurbished, inspected, and cleaned to excellent condition by qualified sellers.
    The item includes original or new accessories and will come in new generic packaging.""",

    'Very Good - Refurbished': """The item shows minimal wear and is backed by a one-year warranty.
    It is fully functional and has been professionally refurbished, inspected, and cleaned to very good condition by qualified sellers.
    The item includes original or new accessories and will come in new generic packaging.""",

    'Good - Refurbished': """The item shows moderate wear and is backed by a one-year warranty.
    It is fully functional and has been professionally refurbished, inspected, and cleaned to good condition by qualified sellers.
    The item includes original or new accessories and will come in a new generic packaging.""",

    'Seller refurbished': """The item has been restored to working order by the eBay seller or a third party.
    This means the item was inspected, cleaned, and repaired to full working order and is in excellent condition.
    This item may or may not be in original packaging.""",

    'Used': """An item that has been used previously.
    The item may have some signs of cosmetic wear but is fully operational and functions as intended.
    This item may be a floor model or store return that has been used.""",

    'For parts or not working': """An item that does not function as intended and is not fully operational.
    This includes items that are defective in ways that render them difficult to use, items that require service or repair, or items missing essential components."""
}

In [15]:
# Creating a function to map condition values to a condition dictionary, which represents labels (e.g: `New`) as keys and description as values (e.g: `A brand-new, unused item...`).
def map_condition(value, condition_dict=condition_replace_map):
    """
    Maps a given condition string to a predefined label and its corresponding description.

    Args:
        value (str): The condition value to be matched.
        condition_dict (dict, optional): A dictionary where keys are condition labels and
                                         values are descriptions. Defaults to `condition_replace_map`.

    Returns:
        tuple: A tuple containing the matched label (str) and its corresponding description (str).
               If no match is found, returns ('No label', 'No description').

    Example:
        >>> map_condition("Excellent - Refurbished device")
        ('Excellent - Refurbished', 'The item is in like-new condition, backed by a one-year warranty. ...')

        >>> map_condition("Unknown condition")
        ('No label', 'No description')
    """
    if isinstance(value, str):
        for label, description in condition_dict.items():
            if label in value:
                return label, description
    return 'No label', 'No description'


In [16]:
# Transforming the dataframe, creating the `condition_label` and `condition_description` variables
df_lazy_conditions = df.lazy().with_columns(
    pl.col('condition')
    # Applying the `map_condition` function, which maps a string to a predefined label and its corresponding description
    .map_elements(
        function=map_condition,
        skip_nulls=False,
        return_dtype=pl.List(pl.Utf8)
    ).alias('new_condition')  # Creating a temporary column to store results
).with_columns(
    pl.col('new_condition').list.get(0).alias('condition_label_clean'),  # Extract label
    pl.col('new_condition').list.get(1).alias('condition_description_clean')  # Extract description
).drop('new_condition')  # Dropping the temporary column

# Visualize the query plan for the transformation
LazyFrame.show_graph(df_lazy_conditions)

In [17]:
# Selecting all unique values for condition, and their soon to be assigned condition labels and condition descriptions
df_lazy_conditions.unique(
    subset='condition',
    maintain_order=True,
).sort(
    by='condition',
    descending=False,
).select(
    ['condition', 'condition_label_clean', 'condition_description_clean']
).collect().to_pandas()

,condition,condition_label_clean,condition_description_clean
0,None,No label,No description
1,Certified - Refurbished: The item is in pristi...,Certified - Refurbished,"The item is in pristine, like-new condition.\n..."
2,Certified - RefurbishedThe item is in pristine...,Certified - Refurbished,"The item is in pristine, like-new condition.\n..."
3,Excellent - Refurbished: The item is in like-n...,Excellent - Refurbished,"The item is in like-new condition, backed by a..."
4,Excellent - RefurbishedThe item is in like-new...,Excellent - Refurbished,"The item is in like-new condition, backed by a..."
5,For parts or not working: An item that does no...,For parts or not working,An item that does not function as intended and...
6,For parts or not workingAn item that does not ...,For parts or not working,An item that does not function as intended and...
7,Good - Refurbished: The item shows moderate we...,Good - Refurbished,The item shows moderate wear and is backed by ...
8,Good - RefurbishedThe item shows moderate wear...,Good - Refurbished,The item shows moderate wear and is backed by ...
9,"New: A brand-new, unused, unopened, undamaged ...",New,"A brand-new, unused, unopened, undamaged item ..."


In [18]:
# Reformatting `condition_label_clean` and `condition_description_clean`: replacing blank spaces for underscores and lowercasing
df = df_lazy_conditions.with_columns(
    pl.col('condition_label_clean')
    .str.to_lowercase()  # Converting the text to lowercase
    .str.replace_all(' - ', '_')  # Replacing hyphen and spaces between words with underscores
    .str.replace_all(' ', '_')  # Replacing spaces with underscores
    .alias('condition_label_clean')
).collect() # Materializing the LazyFrame into a DataFrame

# Selecting all unique values for condition, and their respective reformatted condition labels (condition description is not modified)
df.unique(
    subset='condition',
    maintain_order=True,
).sort(
    by='condition',
    descending=False,
).select(
    ['condition', 'condition_label_clean']
)


condition,condition_label_clean
str,str
null,"""no_label"""
"""Certified - Refurbished: The i…","""certified_refurbished"""
"""Certified - RefurbishedThe ite…","""certified_refurbished"""
"""Excellent - Refurbished: The i…","""excellent_refurbished"""
"""Excellent - RefurbishedThe ite…","""excellent_refurbished"""
…,…
"""Seller refurbishedThe item has…","""seller_refurbished"""
"""Used: An item that has been us…","""used"""
"""UsedAn item that has been used…","""used"""


In [19]:
# Taking a look at the values counts for the `condition_label_clean` column
df['condition_label_clean'].value_counts(
    sort=True,
    parallel=True,
).with_columns(
    (pl.col('count') / df.height).round(2).alias('proportion')
)

condition_label_clean,count,proportion
str,u32,f64
"""used""",2794,0.42
"""new""",1931,0.29
"""no_label""",616,0.09
"""seller_refurbished""",578,0.09
"""for_parts_or_not_working""",173,0.03
"""very_good_refurbished""",151,0.02
"""good_refurbished""",132,0.02
"""excellent_refurbished""",106,0.02
"""open_box""",102,0.02


Observations:
- **New label frequencies**: After cleansing, the most common conditions are `used` (42%) and `new` (29%), which account for the majority of the dataset.

## Transforming the `processor` variable

- TODO : Write analysis about the processor variable

In [20]:
# Taking a look at the values counts for the `condition` column
df['processor'].value_counts(
    sort=True
).with_columns(
    (pl.col('count') / df.height).round(2).alias('proportion')
)

processor,count,proportion
str,u32,f64
null,2783,0.42
"""Does not apply""",531,0.08
"""Intel Core i5 4th generation""",318,0.05
"""Intel Celeron""",183,0.03
"""Intel Core i5 6th generation""",183,0.03
…,…,…
"""Cable""",1,0.0
"""AMD Ryzen 7 5800HS""",1,0.0
"""Intel Core I7 8650u""",1,0.0


In [21]:
processor_replace_map = {
    '?': 'unknown',
    'none': 'unknown',
    'no': 'unknown',
    '^?': 'unknown',
    'does not apply': 'not_applicable'
}

df = df.with_columns(
    pl.col('processor')
    .str.to_lowercase()
    .replace(processor_replace_map)
    .alias('processor_clean')
)

df.select(
    'processor', 'processor_clean'
).head()

processor,processor_clean
str,str
"""Quad Core""","""quad core"""
"""Intel Core i7 8th Gen.""","""intel core i7 8th gen."""
"""Intel Core i5-6300U""","""intel core i5-6300u"""
"""Intel Celeron N""","""intel celeron n"""
"""Intel Core i5 6th Gen.""","""intel core i5 6th gen."""


In [22]:
# Taking a look at unique `processor` with their respective `processor_clean` variables
df.select([
    'processor', 'processor_clean',
]).filter(
    pl.col('processor').str.to_lowercase().is_in(processor_replace_map)
).unique(
    subset='processor',
    maintain_order=True,
).sort(
    by='processor',
    descending=False,
)

processor,processor_clean
str,str
"""?""","""unknown"""
"""DOES NOT APPLY""","""not_applicable"""
"""Does Not Apply""","""not_applicable"""
"""Does not apply""","""not_applicable"""
"""NONE""","""unknown"""
"""No""","""unknown"""
"""^?""","""unknown"""
"""none""","""unknown"""


# Transforming the `color` variable

- `color` is currently a variable with a high number of nulls (approx. 68%)
- `color` contains data in spanish, as evidenced by its values `borgoña` (burgundy), `blanco` (white) or `negro` (black)

In [23]:
# Color value counts and their proportion`
df['color'].value_counts().sort(
    by='count',
    descending=True,
).with_columns(
    (pl.col('count') / df.height).round(2).alias('proportion')
)

color,count,proportion
str,u32,f64
null,4471,0.68
"""Black""",982,0.15
"""Silver""",370,0.06
"""Negro""",264,0.04
"""Gray""",254,0.04
…,…,…
"""Dark Brown""",1,0.0
"""Gray / Silver""",1,0.0
"""Black / Silver""",1,0.0


In [24]:
valid_color_values = ['beige', 'black', 'blue', 'bronze', 'brown', 'burgundy', 'gold', 'gray', 'green', 'grey',
                      'orange', 'pink', 'platinum', 'purple', 'red', 'silver', 'teal', 'white', 'yellow']

colors_translation = {
    'negro': 'black',
    'borgoña': 'burgundy',
    'platino': 'platinum',
    'gris': 'grey',
    'blanco': 'white',
    'plata transparente': 'transparent silver',
}

color_replace_map = {
    'multi': 'multicolor',
    'multi-color': 'multicolor',
    'blk': 'black',
    **colors_translation,
}

color_replace_map

{'multi': 'multicolor',
 'multi-color': 'multicolor',
 'blk': 'black',
 'negro': 'black',
 'borgoña': 'burgundy',
 'platino': 'platinum',
 'gris': 'grey',
 'blanco': 'white',
 'plata transparente': 'transparent silver'}

In [25]:
def clean_color(color: str, color_list: list = valid_color_values) -> str:
    """
    Cleans the input color string by categorizing it based on the presence of valid colors.

    The function checks if the input `color` matches any of the colors in the `color_list`.
    It returns a category based on the number of matches:
    - If the color matches more than one valid color, it returns 'multicolor'.
    - If the color matches exactly one valid color, it returns that color (e.g: 'red').
    - If no valid colors are found, it returns 'other'.

    Args:
        color (str): The input color string to be cleaned and categorized.
        color_list (list, optional): A list of valid colors to check against. Defaults to `valid_color_values`.

    Returns:
        str: The cleaned and categorized color, either a valid color, 'multicolor', or 'other'.
    """
    color_count = sum(1 for c in color_list if c in color.lower())

    if color_count > 1:
        return 'multicolor'
    elif color_count == 1:
        return next((c for c in valid_color_values if c in color.lower()), 'other')
    else:
        return 'other'


In [26]:
# Creates an expression to search for valid colors
valid_color_string_pattern = "|".join(re.escape(color) for color in valid_color_values)

color_reformatted = pl.col('color') \
    .str.to_lowercase() \
    .replace(color_replace_map)

df = df.with_columns(
    pl.coalesce(
        # If color is valid then apply the `clean_color` function, and map its value to a valid color, multicolor or other (rare label, for invalid color categories)
        pl.when(color_reformatted.str.contains(valid_color_string_pattern))
        .then(color_reformatted.map_elements(clean_color, return_dtype=pl.Utf8)),
        # If color value is set to multicolor, then return multicolor
        pl.when(color_reformatted.eq('multicolor'))
        .then(pl.lit('multicolor'))
        # If value is not multicolor nor a single color,then assign it to `other`
        .otherwise(pl.lit('other'))
    ).alias('color_clean')
)

df

brand,price,rating,ratings_count,condition,seller_note,processor,screen_size,manufacturer_color,color,ram_size,ssd_capacity,gpu,processor_speed,type,release_year,maximum_resolution,model,os,features,hard_drive_capacity,country_region_of_manufacture,storage_type,brand_clean,currency_clean,min_price_clean,condition_label_clean,condition_description_clean,processor_clean,color_clean
str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str
"""CHUWI""","""$303.68 """,null,null,"""New: A brand-new, unused, unop…",null,"""Quad Core""","""14 in""",null,"""Gray""","""8 GB""","""1 Used, 1 support Max 1TB M.2…","""Intel Iris Plus Graphics 655""","""Max up to 3.80 GHz""","""Notebook/Laptop""","""2021""","""2160 x 1440""","""CoreBook X""","""Windows 11 Home""","""Backlit Keyboard, Built-in Mi…","""512 GB SSD""","""China""","""SSD (Solid State Drive)""","""chuwi""","""$""",303.68,"""new""","""A brand-new, unused, unopened,…","""quad core""","""gray"""
"""Dell""","""$399.99 to $634.99""",null,null,"""Very Good - RefurbishedThe ite…","""“AAA PCs is a Microsoft Author…","""Intel Core i7 8th Gen.""","""14 in""",null,"""Black""",null,null,"""Intel UHD Graphics 620""","""4.20 GHz (1.90 GHz Base Freque…","""Notebook/Laptop""","""Refurbished in 2023""","""1920 x 1080""","""Dell Latitude 7490""","""Windows 11 Pro""","""Backlit Keyboard, Bluetooth, B…","""2 TB""",null,"""SSD (Solid State Drive)""","""dell""","""$""",399.99,"""very_good_refurbished""","""The item shows minimal wear an…","""intel core i7 8th gen.""","""black"""
"""Dell""","""$175.00 """,null,null,"""UsedAn item that has been used…","""“Well kept, fully functional, …","""Intel Core i5-6300U""","""14 in""",null,"""Black""","""16 GB""","""500 GB""","""Intel HD Graphics""","""2.40 GHz""","""Notebook/Laptop""","""2019""","""1920 x 1080""","""Dell Latitude E5470""","""Windows 10 Pro""","""10/100 LAN Card, Backlit Keybo…","""500 GB""","""China""","""SSD (Solid State Drive)""","""dell""","""$""",175.0,"""used""","""An item that has been used pre…","""intel core i5-6300u""","""black"""
"""HP""","""$84.99 """,null,null,"""Good - RefurbishedThe item sho…","""“1-Year Allstate warranty. The…","""Intel Celeron N""","""11.6 in""",null,"""Black""","""4 GB""",null,"""Intel HD Graphics 500""","""2.40 GHz""","""Notebook/Laptop""",null,"""1366 x 768""","""HP Chromebook 11 G6""","""Chrome OS""","""Bluetooth, Built-in Microphone…","""16 GB""",null,"""eMMC""","""hp""","""$""",84.99,"""good_refurbished""","""The item shows moderate wear a…","""intel celeron n""","""black"""
"""Dell""","""$101.22 """,null,null,"""Good - RefurbishedThe item sho…","""“Laptops is tested & fully wor…","""Intel Core i5 6th Gen.""","""Minimum 12.5""""",null,null,"""8 GB""","""256 GB""","""Integrated""","""Minimum 1.40 GHz""","""Notebook/Laptop""","""2015""","""1366 x 768""","""Various Models""","""Windows 10""","""10/100 LAN Card, Built-in Micr…","""NO HDD""",null,"""SSD (Solid State Drive)""","""dell""","""$""",101.22,"""good_refurbished""","""The item shows moderate wear a…","""intel core i5 6th gen.""","""other"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
null,"""$108.06 """,null,null,"""New: A brand-new, unused, unop…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""unknown""","""$""",108.06,"""new""","""A brand-new, unused, unopened,…",null,"""other"""
null,"""$2,108.99 """,null,null,"""Seller refurbished: The item h…",null,null,null,null,"""Negro""",null,null,"""Gráficos Intel UHD 620""",null,null,null,null,null,null,null,null,null,null,"""unknown""","""$""",2108.99,"""seller_refurbished""","""The item has been restored to …",null,"""black"""
null,"""$105.86 """,null,null,"""New: A brand-new, unused, unop…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""unknown""","""$""",105.86,"""new""","""A brand-new, unused, unopened,…",null,"""other"""


In [27]:
# Taking a look at the newly transformed color cleansed labels
df.select([
    'color', 'color_clean',
]).unique(
    subset='color',
    maintain_order=True,
).sort(
    by='color',
).transpose(
    include_header=True,
    header_name='column_name',
)

column_name,column_0,column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10,column_11,column_12,column_13,column_14,column_15,column_16,column_17,column_18,column_19,column_20,column_21,column_22,column_23,column_24,column_25,column_26,column_27,column_28,column_29,column_30,column_31,column_32,column_33,column_34,column_35,…,column_46,column_47,column_48,column_49,column_50,column_51,column_52,column_53,column_54,column_55,column_56,column_57,column_58,column_59,column_60,column_61,column_62,column_63,column_64,column_65,column_66,column_67,column_68,column_69,column_70,column_71,column_72,column_73,column_74,column_75,column_76,column_77,column_78,column_79,column_80,column_81,column_82
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""color""",null,"""Abyss Blue""","""Arctic Grey""","""BLACK""","""BLK""","""Beige""","""Black""","""Black / Gray""","""Black / Grey""","""Black / Silver""","""Black/ Blue / Sandtone / Plati…","""Black/Blue""","""Black/Gray""","""Black/Red""","""Blanco""","""Blue""","""Bronze""","""Brown""","""Carbon Black""","""Charcoal Gray""","""Cloud Grey""","""Dark Brown""","""Dark Grey""","""Dark Metallic Moon""","""Eclipse Gray""","""Fog Blue Aluminum""","""Gold""","""Gray""","""Gray / Silver""","""Gray and Black""","""Gray, Blue""","""Gray, Granite Gray""","""Gray, Platinum Silver""","""Green""","""Grey""","""Grey/Black""",…,"""Multicolor""","""Natural Silver""","""Negro""","""Orange""","""Pink""","""Plata Transparente""","""Platino""","""Platinum""","""Platinum Silver""","""Pure Silver""","""Purple""","""Quiet Blue""","""Red""","""STEEL GRAY""","""Sandstone""","""See Photos""","""Silver""","""Silver & Black""","""Silver + Black""","""Silver / Black""","""Silver and Black""","""Silver, Black""","""Silver, Metallic Silver""","""Silver, Platinum Silver""","""Silver/Black""","""Space Gray""","""Standard""","""Steam Blue""","""Storm Gray""","""Teal""","""Tech Black""","""Warm Gold""","""White""","""Yellow""","""black""","""borgoña""","""see pictures"""
"""color_clean""","""other""","""blue""","""grey""","""black""","""black""","""beige""","""black""","""multicolor""","""multicolor""","""multicolor""","""multicolor""","""multicolor""","""multicolor""","""multicolor""","""white""","""blue""","""bronze""","""brown""","""black""","""gray""","""grey""","""brown""","""grey""","""other""","""gray""","""blue""","""gold""","""gray""","""multicolor""","""multicolor""","""multicolor""","""gray""","""multicolor""","""green""","""grey""","""multicolor""",…,"""multicolor""","""silver""","""black""","""orange""","""pink""","""silver""","""platinum""","""platinum""","""multicolor""","""silver""","""purple""","""blue""","""red""","""gray""","""other""","""other""","""silver""","""multicolor""","""multicolor""","""multicolor""","""multicolor""","""multicolor""","""silver""","""multicolor""","""multicolor""","""gray""","""other""","""blue""","""gray""","""teal""","""black""","""gold""","""white""","""yellow""","""black""","""burgundy""","""other"""
